# TOPIC 10 – ĐO LƯỜNG RỦI RO DANH MỤC VN30 BẰNG DCC-GARCH
## Thành viên 3 – Lâm: DCC-GARCH, tương quan động
Chạy **sau** TV2. Chạy bằng **Restart Kernel → Run All**. Thứ tự chung của cả nhóm: TV1 → TV2 → TV3 → TV4 → TV5.

| | |
|---|---|
| Đọc từ | `TV1_Data_EDA/log_returns.csv`, `TV2_GARCH/std_resid.csv`, `cond_vol.csv`, `garch_selected_models.csv` |
| Ghi ra | `TV3_DCC/` (`dcc_matrices.npz`, `TV3_dcc_results.xlsx`, hình) |
| Cấu hình | ô 0.2 (đường dẫn, hằng số) – **giống hệt ở cả 5 notebook**, chỉ cần sửa `PROJECT_PATH` / `OUTPUT_PATH` cho đúng máy |

Thư viện: numpy, pandas, scipy, matplotlib, openpyxl (statsmodels tuỳ chọn). Xem `README.md`.

In [ ]:
# ===================== 0.1 THƯ VIỆN =====================
import os                     # đọc biến môi trường
import re                     # biểu thức chính quy: tách dòng cổ tức / chia tách trong file Yahoo
import warnings               # tắt cảnh báo không cần thiết
import datetime as dt         # kiểu ngày tháng
from itertools import combinations   # tạo các cặp cổ phiếu
from pathlib import Path      # thao tác đường dẫn an toàn trên Windows

import numpy as np            # tính toán ma trận
import pandas as pd           # bảng dữ liệu
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats                     # phân phối Normal / t / chi-bình phương, JB
from scipy.optimize import minimize         # tối đa hoá hàm hợp lý (MLE)
from scipy.signal import lfilter            # lọc tuyến tính nhanh cho phương trình GARCH
from scipy.special import gammaln           # log Gamma cho mật độ Student-t

try:                                        # statsmodels: chỉ dùng cho ADF / KPSS (nếu có)
    from statsmodels.tsa.stattools import adfuller, kpss
    from statsmodels.tools.sm_exceptions import InterpolationWarning
    HAS_SM = True
except ImportError:
    HAS_SM = False

try:                                        # display() có sẵn trong Jupyter; chạy file .py thì thay bằng print
    display
except NameError:
    display = print

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 100
print("statsmodels có sẵn:", HAS_SM)

In [ ]:
# ===================== 0.2 CẤU HÌNH DÙNG CHUNG CHO CẢ 5 THÀNH VIÊN =====================
# >>> ĐƯỜNG DẪN – sửa 2 dòng này nếu đổi máy / đổi thư mục <<<
PROJECT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo"          # thư mục dự án (chứa input/, ouput/, code/)
OUTPUT_PATH = r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\bản demo\ouput"    # nơi lưu toàn bộ kết quả


def find_base_dir() -> Path:
    """Tìm thư mục dự án: biến môi trường PROJECT_DIR -> PROJECT_PATH -> thư mục cha của code/."""
    env = os.environ.get("PROJECT_DIR")
    if env:
        return Path(env)
    if Path(PROJECT_PATH).exists():
        return Path(PROJECT_PATH)
    cwd = Path.cwd()
    return cwd.parent if cwd.name.lower() == "code" else cwd


def pick_dir(base: Path, names) -> Path:
    """Chọn thư mục con đã tồn tại (chấp nhận Input/input)."""
    return next((base / n for n in names if (base / n).exists()), base / names[0])


BASE_DIR = find_base_dir()
INPUT_DIR = pick_dir(BASE_DIR, ["input", "Input"])
if not list(INPUT_DIR.glob("*.xlsx")):                  # dự phòng: thư mục Input cũ của Project midterm
    old_in = Path(r"D:\Năm 3\Ứng dụng công nghệ trong TCNH\Project midterm\Input")
    if list(old_in.glob("*.xlsx")):
        INPUT_DIR = old_in
# Output: biến môi trường OUTPUT_DIR -> OUTPUT_PATH (nếu ổ D: tồn tại) -> BASE_DIR/ouput
OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR") or
                  (OUTPUT_PATH if Path(OUTPUT_PATH).anchor and Path(Path(OUTPUT_PATH).anchor).exists()
                   else BASE_DIR / "ouput"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ROOT_OUT = OUTPUT_DIR                                    # mỗi thành viên một thư mục con trong ouput/
OUT = {k: ROOT_OUT / v for k, v in {"TV1": "TV1_Data_EDA", "TV2": "TV2_GARCH", "TV3": "TV3_DCC",
                                     "TV4": "TV4_VaR_Backtest", "TV5": "TV5_Euler_Stress"}.items()}
for d in OUT.values():
    d.mkdir(parents=True, exist_ok=True)

# ---- Hằng số dùng chung ----
TICKERS_ORDER = ["FPT", "HPG", "MWG", "VCB", "VHM"]    # thứ tự hiển thị (mã khác trong Input vẫn được đọc)
ALPHA = 0.05                 # mức ý nghĩa của mọi kiểm định
LB_LAGS = [5, 10, 20]        # số trễ Ljung-Box
TRADING_DAYS = 252           # số phiên / năm để năm hoá
SCALE = 100.0                # ước lượng GARCH trên lợi suất % (r × 100) cho ổn định số học
SELECT_BY = "BIC"            # tiêu chí chọn mô hình GARCH: "AIC" hoặc "BIC"
PAIRS = [("VCB", "VHM"), ("HPG", "FPT")]               # cặp tương quan động theo đề
VAR_LEVELS = [0.95, 0.99]    # mức tin cậy VaR
HS_WINDOW = 250              # cửa sổ Historical Simulation (≈ 1 năm)
BASEL_WINDOW = 250           # cửa sổ Basel Traffic Light (250 phiên gần nhất)
VIOL_MODEL = "Normal"        # >>> MÔ HÌNH ĐÁNH DẤU VI PHẠM TRÊN MỌI HÌNH & DIỄN GIẢI TV4: "Normal" hoặc "t" <<<
VIOL_OOS = {"Normal": "DCC-N", "t": "DCC-t"}[VIOL_MODEL]   # tên tương ứng trong phần ngoài mẫu
EULER_CL = 0.99              # mức tin cậy dùng cho phân bổ rủi ro Euler
STRESS = {                   # giai đoạn khủng hoảng theo đề (TV5)
    "Covid-19 (02–04/2020)": ("2020-02-01", "2020-04-30"),
    "Trái phiếu & lãi suất (04–11/2022)": ("2022-04-01", "2022-11-30"),
}


STRESS_COLORS = ["tab:red", "tab:orange"]   # mỗi giai đoạn một màu để phân biệt Covid-19 với Trái phiếu & lãi suất


def shade_stress(ax, alpha=0.12, label=True, xlim=None):
    """Tô màu từng giai đoạn khủng hoảng (mỗi giai đoạn một màu, chú thích theo tên giai đoạn).
    xlim=(đầu, cuối): bỏ qua giai đoạn nằm hoàn toàn ngoài khoảng này (tránh kéo giãn trục thời gian)."""
    for (name, (s, e)), c in zip(STRESS.items(), STRESS_COLORS):
        s, e = pd.Timestamp(s), pd.Timestamp(e)
        if xlim is not None and (e < xlim[0] or s > xlim[1]):
            continue
        ax.axvspan(s, e, color=c, alpha=alpha, label=(name if label else None))


def grid_axes(n, ncol=2, h=3.2, w=13, **kw):
    """Tạo lưới ô vẽ cho n mã, ẩn các ô thừa."""
    ncol = min(ncol, n)
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(w, h * nrow), squeeze=False, **kw)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, axes.flat


print("BASE_DIR  :", BASE_DIR)
print("INPUT_DIR :", INPUT_DIR, "| tồn tại:", INPUT_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


def period_masks(index):
    """Mặt nạ True/False cho từng giai đoạn: các giai đoạn STRESS + 'Bình thường' (ngoài mọi giai đoạn stress)."""
    m = {name: (index >= s) & (index <= e) for name, (s, e) in STRESS.items()}
    m = {"Bình thường": ~np.any(np.column_stack(list(m.values())), axis=1), **m}
    return m


## Nạp kết quả của các thành viên trước

In [ ]:
# ===================== NẠP KẾT QUẢ TV1, TV2 + HÀM DÙNG CHUNG =====================
def need(path, prev):
    """Báo lỗi rõ ràng nếu chưa chạy notebook của thành viên trước."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Chưa có {path} – hãy chạy notebook {prev} trước (xem README.md).")
    return Path(path)


log_ret = pd.read_csv(need(OUT["TV1"] / "log_returns.csv", "TV1"), index_col=0, parse_dates=True, float_precision="round_trip")
log_ret.index.name = "Date"
tickers = list(log_ret.columns); N = len(tickers); T = len(log_ret)
print(f"[TV1] log_ret: {T} phiên × {N} mã ({', '.join(tickers)}) | {log_ret.index.min().date()} -> {log_ret.index.max().date()}")
std_resid = pd.read_csv(need(OUT["TV2"] / "std_resid.csv", "TV2"), index_col=0, parse_dates=True, float_precision="round_trip")
cond_vol = pd.read_csv(need(OUT["TV2"] / "cond_vol.csv", "TV2"), index_col=0, parse_dates=True, float_precision="round_trip")
sel_h = pd.read_csv(need(OUT["TV2"] / "garch_selected_models.csv", "TV2"), index_col=0, float_precision="round_trip").loc[tickers]
best_name = sel_h["Mô hình"]                                                       # tên mô hình biên đã chọn
SPEC = {tk: dict(gjr=bool(sel_h.loc[tk, "gjr"]), dist=str(sel_h.loc[tk, "dist"])) for tk in tickers}
mu_vec = sel_h["mu"].astype(float)                                                  # μ thập phân
sigma_fcst = sel_h["sigma_T+1"].astype(float).rename("sigma_T+1")
selected = {tk: {"k": int(sel_h.loc[tk, "k"]), "converged": bool(sel_h.loc[tk, "converged"])} for tk in tickers}   # bản rút gọn
print("[TV2] mô hình biên:", best_name.to_dict())

# Sai số chuẩn sandwich (định nghĩa ở TV2 mục 2.1, dùng lại để ước lượng sai số a, b của DCC)
def robust_se(ll_fun, x, h=1e-4):
    """Sai số chuẩn sandwich A⁻¹BA⁻¹ (A: Hessian số, B: tích ngoài của score) – Bollerslev & Wooldridge (1992)."""
    x = np.asarray(x, float)
    k = len(x)
    step = h * np.maximum(np.abs(x), 1e-2)
    scores = np.empty((len(ll_fun(x)), k))
    for i in range(k):
        e = np.zeros(k); e[i] = step[i]
        scores[:, i] = (ll_fun(x + e) - ll_fun(x - e)) / (2 * step[i])
    L = lambda p: ll_fun(p).sum()
    Hs = np.empty((k, k))
    for i in range(k):
        for j in range(i, k):
            ei = np.zeros(k); ei[i] = step[i]
            ej = np.zeros(k); ej[j] = step[j]
            Hs[i, j] = Hs[j, i] = (L(x + ei + ej) - L(x + ei - ej) - L(x - ei + ej) + L(x - ei - ej)) / (4 * step[i] * step[j])
    try:
        A_inv = np.linalg.inv(-Hs)
        return np.sqrt(np.clip(np.diag(A_inv @ (scores.T @ scores) @ A_inv), 0, None))
    except np.linalg.LinAlgError:
        return np.full(k, np.nan)

---
# THÀNH VIÊN 3: MÔ HÌNH HOÁ TƯƠNG QUAN ĐỘNG (DCC-GARCH ESTIMATION)
## 3.1 Ước lượng DCC(1,1) – Engle (2002)
**Tầng 1:**
- a. $Q_t = (1-a-b)\bar{Q} + a\,\epsilon_{t-1}\epsilon_{t-1}' + b\,Q_{t-1}$, với $\bar Q$ = hiệp phương sai mẫu của ε (correlation targeting).
- b. Điều kiện dừng: $a \ge 0,\ b \ge 0,\ a + b < 1$.
- c. $R_t = \operatorname{diag}(Q_t)^{-1/2}\,Q_t\,\operatorname{diag}(Q_t)^{-1/2}$.
- d. $H_t = D_t R_t D_t$, $D_t = \operatorname{diag}(\sigma_{1,t}, \dots, \sigma_{N,t})$ lấy từ TV2.
- Log-likelihood phần tương quan (bước 2): $LL_C(a,b) = -\tfrac12\sum_t\left(\ln|R_t| + \epsilon_t'R_t^{-1}\epsilon_t - \epsilon_t'\epsilon_t\right)$.

**Tầng 2:** Input `std_resid` (ε), `cond_vol` (σ), `mu_vec` của TV2. Output dùng chung: `a_hat`, `b_hat`, `R_t`, `H_t` (T × N × N),
`R_fcst`, `H_fcst` (dự báo phiên T+1). Tối ưu bằng SLSQP; sai số chuẩn robust (chưa hiệu chỉnh sai số ước lượng bước 1).

**Tầng 3:** a nhỏ (0,01–0,05), b lớn (> 0,9) là điển hình; half-life tương quan $= \ln 0{,}5/\ln(a+b)$ phiên.

In [ ]:
# ===================== 3.1 DCC(1,1) =====================
Z = std_resid.values                                   # T × N
Qbar = Z.T @ Z / T


def dcc_filter(a, b, Z, Qbar):
    """Trả về Q (T×N×N), R (T×N×N) và Q_{T+1}."""
    Tn, Nn = Z.shape
    Q = np.empty((Tn, Nn, Nn)); R = np.empty((Tn, Nn, Nn))
    q = Qbar.copy()
    for t in range(Tn):
        if t > 0:
            q = (1 - a - b) * Qbar + a * np.outer(Z[t - 1], Z[t - 1]) + b * q
        Q[t] = q
        d = 1 / np.sqrt(np.diag(q))
        R[t] = q * np.outer(d, d)                      # diag(Q)^-1/2 Q diag(Q)^-1/2
    q_next = (1 - a - b) * Qbar + a * np.outer(Z[-1], Z[-1]) + b * q
    return Q, R, q_next


def corr_loglik_t(R, Z):
    """Đóng góp log-likelihood phần tương quan theo từng ngày."""
    _, logdet = np.linalg.slogdet(R)
    quad = np.einsum("ti,tij,tj->t", Z, np.linalg.inv(R), Z)
    return -0.5 * (logdet + quad - np.einsum("ti,ti->t", Z, Z))


def dcc_nll(p):
    a, b = p
    if a < 0 or b < 0 or a + b >= 1:
        return 1e10
    return -corr_loglik_t(dcc_filter(a, b, Z, Qbar)[1], Z).sum()


best = None
for x0 in [(0.02, 0.95), (0.05, 0.90), (0.10, 0.80)]:
    res = minimize(dcc_nll, x0, method="SLSQP", bounds=[(0, 1), (0, 1)],
                   constraints=[{"type": "ineq", "fun": lambda p: 0.9999 - p[0] - p[1]}],
                   options={"maxiter": 500, "ftol": 1e-10})
    if best is None or res.fun < best.fun:
        best = res
a_hat, b_hat = best.x
se_ab = robust_se(lambda p: corr_loglik_t(dcc_filter(p[0], p[1], Z, Qbar)[1], Z), best.x)
Q_t, R_t, Q_next = dcc_filter(a_hat, b_hat, Z, Qbar)
half_life_corr = np.log(0.5) / np.log(a_hat + b_hat)

dcc_params = pd.DataFrame({"Ước lượng": [a_hat, b_hat, a_hat + b_hat, half_life_corr],
                           "Std.Err": [se_ab[0], se_ab[1], np.nan, np.nan]},
                          index=["a", "b", "a + b", "Half-life tương quan (phiên)"])
dcc_params["p-value"] = 2 * stats.norm.sf(np.abs(dcc_params["Ước lượng"] / dcc_params["Std.Err"]))
display(dcc_params.round(5))
ok_stat = (a_hat >= 0) and (b_hat >= 0) and (a_hat + b_hat < 1)
print(f"Điều kiện dừng a ≥ 0, b ≥ 0, a + b < 1: {'THOẢ' if ok_stat else 'KHÔNG THOẢ'} "
      f"(a = {a_hat:.4f}, b = {b_hat:.4f}, a + b = {a_hat + b_hat:.4f}) | hội tụ: {best.success}")

In [ ]:
# ===================== 3.1c–d R_t, H_t VÀ DỰ BÁO T+1 =====================
S = cond_vol.values                                     # σ_t (thập phân)
H_t = R_t * S[:, :, None] * S[:, None, :]               # H_t = D_t R_t D_t
d_next = 1 / np.sqrt(np.diag(Q_next))
R_fcst = Q_next * np.outer(d_next, d_next)
H_fcst = R_fcst * np.outer(sigma_fcst.values, sigma_fcst.values)
eig_min = np.linalg.eigvalsh(H_t).min()
print(f"Đường chéo R_t = 1: {np.allclose(np.einsum('tii->ti', R_t), 1)} | H_t đối xứng: {np.allclose(H_t, H_t.transpose(0, 2, 1))} "
      f"| H_t xác định dương (λ_min = {eig_min:.2e}): {eig_min > 0}")
print(f"Kiểm tra sqrt(diag H_t) = σ_t của TV2: {np.allclose(np.sqrt(np.einsum('tii->ti', H_t)), S)}")
idx_tk = {tk: i for i, tk in enumerate(tickers)}
pairs_all = list(combinations(tickers, 2))
dyn_corr = pd.DataFrame({f"{x}-{y}": R_t[:, idx_tk[x], idx_tk[y]] for x, y in pairs_all}, index=log_ret.index)
print("Ma trận tương quan dự báo R_{T+1}:")
display(pd.DataFrame(R_fcst, index=tickers, columns=tickers).round(3))

## 3.2 So sánh mô hình: DCC vs CCC vs Hiệp phương sai tĩnh
**Tầng 1:** Cả 3 mô hình được so sánh bằng **cùng một hàm hợp lý chuẩn đa biến**
$LL = -\tfrac12\sum_t\left[N\ln 2\pi + \ln|H_t| + e_t'H_t^{-1}e_t\right]$, $e_t = r_t - \mu$:
- **Static:** $H_t = \Sigma$ cố định (hiệp phương sai mẫu) – bỏ qua cả biến động lẫn tương quan thay đổi.
- **CCC (Bollerslev, 1990):** $H_t = D_t R D_t$, R cố định = tương quan mẫu của ε.
- **DCC:** $H_t = D_t R_t D_t$.

CCC và DCC dùng chung $D_t$ của TV2 nên chênh lệch LL chỉ đến từ việc cho tương quan thay đổi. Kiểm định LR: $H_0$: a = b = 0 (DCC thu về CCC),
$LR = 2(LL_{DCC} - LL_{CCC})$ ~ hỗn hợp $\tfrac14\chi^2_0 + \tfrac12\chi^2_1 + \tfrac14\chi^2_2$ (tham số nằm trên biên).

**Tầng 3:** DCC có LL lớn nhất và AIC/BIC nhỏ nhất, LR p < 0,05 ⇒ **tương quan giữa các cổ phiếu VN30 thay đổi theo thời gian**, không bất biến.

In [ ]:
# ===================== 3.2 BENCHMARK =====================
e = (log_ret - mu_vec).values


def mvn_loglik_from_R(R):
    """LL chuẩn đa biến khi H_t = D_t R_t D_t (R: N×N hoặc T×N×N)."""
    R = np.broadcast_to(R, (T, N, N))
    _, logdet_R = np.linalg.slogdet(R)
    quad = np.einsum("ti,tij,tj->t", Z, np.linalg.inv(R), Z)
    return np.sum(-0.5 * (N * np.log(2 * np.pi) + 2 * np.log(S).sum(axis=1) + logdet_R + quad))


k_margin = int(sum(selected[tk]["k"] for tk in tickers))
R_ccc = np.corrcoef(Z, rowvar=False)
Sigma = np.cov(log_ret.values, rowvar=False, bias=True)
bench = pd.DataFrame({
    "LogLik": [stats.multivariate_normal(log_ret.mean().values, Sigma).logpdf(log_ret.values).sum(),
               mvn_loglik_from_R(R_ccc), mvn_loglik_from_R(R_t)],
    "k": [N + N * (N + 1) // 2, k_margin + N * (N - 1) // 2, k_margin + N * (N - 1) // 2 + 2]},
    index=["Static Covariance", "CCC-GARCH", "DCC-GARCH"])
bench["AIC"] = -2 * bench["LogLik"] + 2 * bench["k"]
bench["BIC"] = -2 * bench["LogLik"] + np.log(T) * bench["k"]
bench["Hạng AIC"] = bench["AIC"].rank().astype(int)
bench["Hạng BIC"] = bench["BIC"].rank().astype(int)
display(bench.round(2))
LR_dcc = 2 * (bench.loc["DCC-GARCH", "LogLik"] - bench.loc["CCC-GARCH", "LogLik"])
p_lr = 0.5 * stats.chi2.sf(LR_dcc, 1) + 0.25 * stats.chi2.sf(LR_dcc, 2)
print(f"LR (DCC vs CCC) = {LR_dcc:.2f}, p-value = {p_lr:.3g} -> "
      + ("Bác bỏ H0: tương quan THAY ĐỔI theo thời gian" if p_lr < ALPHA else "Chưa bác bỏ tương quan bất biến"))

## 3.3 Trực quan hoá tương quan động
**Tầng 3:** đường xanh = ρ_t của DCC; nét đứt = tương quan CCC (hằng số); đường xanh lá = tương quan trượt 60 phiên (Pearson) để đối chiếu.
Vùng tô đỏ = giai đoạn khủng hoảng. Tương quan tăng vọt trong khủng hoảng ⇒ lợi ích đa dạng hoá giảm đúng lúc cần nhất.

**Lưu ý khi đọc các cú giảm đột ngột:** DCC phản ứng với tích $\epsilon_{i,t-1}\epsilon_{j,t-1}$. Một ngày hai mã có phần dư rất lớn **trái dấu**
kéo tương quan xuống mạnh. Ví dụ 09/07/2021: HPG −10% (ε = −5,1) và MWG +11,8% (ε = +6,2) – cả hai là lợi suất **dồn qua 4 ngày Yahoo thiếu
dữ liệu** (xem `outliers_check.csv` của TV1) ⇒ HPG–MWG rơi xuống ≈ −0,40 ngày 13/07/2021 là **hiệu ứng của dữ liệu**, không phải thay đổi
kinh tế thật. Ngược lại, cú giảm của các cặp có VCB ngày 24–25/01/2022 là thật: VCB tăng +4,2% trong phiên thị trường giảm mạnh.

In [ ]:
# ===================== 3.3 ĐỒ THỊ TƯƠNG QUAN ĐỘNG =====================
pairs_main = [p for p in PAIRS if p[0] in idx_tk and p[1] in idx_tk]
fig, axes = plt.subplots(len(pairs_main), 1, figsize=(13, 3.8 * len(pairs_main)), sharex=True, squeeze=False)
for ax, (x, y) in zip(axes.flat, pairs_main):
    s = pd.Series(R_t[:, idx_tk[x], idx_tk[y]], index=log_ret.index)
    shade_stress(ax)
    ax.plot(s.index, s, lw=1, color="tab:blue", label="DCC ρ_t")
    ax.plot(s.index, log_ret[x].rolling(60).corr(log_ret[y]), lw=0.8, color="tab:green", alpha=0.7, label="Pearson trượt 60 phiên")
    ax.axhline(R_ccc[idx_tk[x], idx_tk[y]], color="black", ls="--", lw=1, label="CCC (hằng số)")
    ax.set_title(f"Tương quan động {x} – {y} | TB = {s.mean():.3f}, min = {s.min():.3f}, max = {s.max():.3f}")
    ax.grid(alpha=0.3); ax.legend(fontsize=8, ncol=4, loc="lower left")
fig.suptitle(f"DCC-GARCH(1,1): a = {a_hat:.4f}, b = {b_hat:.4f}", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV3"] / "3_dcc_main_pairs.png", dpi=150, bbox_inches="tight"); plt.show()

fig, axes = grid_axes(len(pairs_all), h=2.4, sharex=True, sharey=True)
for ax, col in zip(axes, dyn_corr.columns):
    x, y = col.split("-")
    shade_stress(ax, label=False)
    ax.plot(dyn_corr.index, dyn_corr[col], lw=0.8)
    ax.axhline(R_ccc[idx_tk[x], idx_tk[y]], color="black", ls="--", lw=0.8)
    ax.set_title(col, fontsize=10); ax.grid(alpha=0.3)
fig.suptitle("Tương quan động DCC của tất cả các cặp", fontsize=13, y=1.0)
plt.tight_layout(); plt.savefig(OUT["TV3"] / "3_dcc_all_pairs.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# ===================== 3.3b TƯƠNG QUAN THEO GIAI ĐOẠN =====================
masks = period_masks(log_ret.index)
avg_corr = dyn_corr.mean(axis=1)
corr_by_period = pd.DataFrame({name: dyn_corr[m].mean() for name, m in masks.items()})
corr_by_period.loc["TRUNG BÌNH CÁC CẶP"] = corr_by_period.mean()
display(corr_by_period.round(3))

fig, ax = plt.subplots(figsize=(13, 4))
shade_stress(ax)
ax.plot(avg_corr.index, avg_corr, color="tab:purple", lw=1.1, label="Tương quan DCC trung bình các cặp")
ax.axhline(avg_corr.mean(), color="black", ls="--", lw=0.9, label=f"Trung bình mẫu = {avg_corr.mean():.3f}")
ax.set_title("Tương quan trung bình của 5 mã theo thời gian"); ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(OUT["TV3"] / "3_dcc_average_corr.png", dpi=150); plt.show()

# ---- Lưu kết quả TV3 ----
# dates/tickers lưu dạng chuỗi Unicode cố định (U10) để đọc lại được bằng np.load(..., allow_pickle=False)
np.savez_compressed(OUT["TV3"] / "dcc_matrices.npz",
                    dates=np.array(log_ret.index.strftime("%Y-%m-%d").tolist(), dtype="U10"),
                    tickers=np.array(tickers, dtype="U10"), R=R_t, H=H_t, R_fcst=R_fcst, H_fcst=H_fcst, a=a_hat, b=b_hat, Qbar=Qbar, mu=mu_vec.values)
with pd.ExcelWriter(OUT["TV3"] / "TV3_dcc_results.xlsx", engine="openpyxl") as xw:
    dcc_params.to_excel(xw, sheet_name="Tham_so_DCC")
    bench.to_excel(xw, sheet_name="Benchmark")
    corr_by_period.to_excel(xw, sheet_name="Tuong_quan_theo_giai_doan")
    pd.DataFrame(R_fcst, index=tickers, columns=tickers).to_excel(xw, sheet_name="R_T+1")
    pd.DataFrame(H_fcst, index=tickers, columns=tickers).to_excel(xw, sheet_name="H_T+1")
    pd.DataFrame(Qbar, index=tickers, columns=tickers).to_excel(xw, sheet_name="Q_bar")
    dyn_corr.to_excel(xw, sheet_name="Tuong_quan_dong")
print("✔ Đã lưu kết quả TV3 vào", OUT["TV3"])

In [ ]:
# ===================== 3.3c HEATMAP THEO GIAI ĐOẠN & ĐỒ THỊ CHỒNG TẤT CẢ CÁC CẶP =====================
fig, axes = plt.subplots(1, len(masks), figsize=(4.4 * len(masks) + 1.0, 4.0))
for ax, (name, mk) in zip(np.atleast_1d(axes), masks.items()):
    Rm = R_t[np.asarray(mk)].mean(axis=0)                  # ma trận tương quan DCC trung bình trong giai đoạn
    im = ax.imshow(Rm, cmap="Reds", vmin=0, vmax=1)
    ax.set_xticks(range(N)); ax.set_xticklabels(tickers); ax.set_yticks(range(N)); ax.set_yticklabels(tickers)
    for i in range(N):
        for j in range(N):
            ax.text(j, i, f"{Rm[i, j]:.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(name, fontsize=10)
fig.colorbar(im, ax=list(np.atleast_1d(axes)), fraction=0.025, pad=0.02)
plt.savefig(OUT["TV3"] / "3_dcc_heatmap_by_period.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(13, 5))
shade_stress(ax)
for col in dyn_corr.columns:
    ax.plot(dyn_corr.index, dyn_corr[col], lw=0.6, alpha=0.5)
ax.plot(avg_corr.index, avg_corr, color="black", lw=2, label="Tương quan trung bình các cặp")
ax.set_title("Tương quan động DCC – tất cả các cặp"); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.savefig(OUT["TV3"] / "3_dcc_all_pairs_overlay.png", dpi=150); plt.show()

## 3.4 Diễn giải cốt lõi tham số a và b

In [ ]:
# ===================== 3.4 DIỄN GIẢI TỰ ĐỘNG =====================
print(f"a = {a_hat:.4f}: mỗi phiên, tích chéo cú sốc ε_(t-1)ε_(t-1)' đóng góp {a_hat:.1%} vào Q_t -> "
      + ("tương quan phản ứng NHẸ với một cú sốc đơn lẻ; chỉ tăng mạnh khi cú sốc chung lặp lại nhiều phiên."
         if a_hat < 0.05 else "tương quan phản ứng KHÁ MẠNH với cú sốc mới."))
print(f"b = {b_hat:.4f}: {b_hat:.1%} của Q_(t-1) được giữ lại -> tương quan có 'trí nhớ' dài, thay đổi chậm và mượt.")
print(f"a + b = {a_hat + b_hat:.4f}: half-life ≈ {half_life_corr:.1f} phiên (~{half_life_corr / 21:.1f} tháng) -> "
      "sau một cú sốc chung, tương quan duy trì ở mức cao nhiều tuần rồi mới từ từ quay về Q̄.")
r_norm, r_cr = corr_by_period.loc["TRUNG BÌNH CÁC CẶP"].iloc[0], corr_by_period.loc["TRUNG BÌNH CÁC CẶP"].iloc[1:]
print("Tương quan TB: bình thường = " + f"{r_norm:.3f}; " + "; ".join(f"{k} = {v:.3f}" for k, v in r_cr.items()))

**Ý nghĩa kinh tế:**
- **a – phản ứng với cú sốc:** nếu hôm qua hai mã cùng có phần dư lớn cùng dấu (cùng giảm sâu), $Q_t$ tăng thêm $a\,\epsilon_{i,t-1}\epsilon_{j,t-1}$
  ⇒ tương quan hôm nay tăng. a nhỏ ⇒ một phiên riêng lẻ không làm tương quan đảo chiều ngay.
- **b – tính bền vững:** trọng số của $Q_{t-1}$; b lớn ⇒ cấu trúc tương quan thay đổi chậm.
- **a + b gần 1:** lực kéo về $\bar Q$ (hệ số 1 − a − b) rất yếu ⇒ cú sốc tương quan kéo dài nhiều tuần; lợi ích đa dạng hoá giảm trong khủng hoảng
  và **không phục hồi ngay** khi thị trường ổn định ⇒ VaR và tỷ trọng danh mục phải dùng $H_t$ động thay vì ma trận tĩnh.

### TV3 – Gợi ý trả lời vấn đáp
**Câu 1 – Vì sao không dùng tương quan Pearson thông thường?** Pearson là **một con số trung bình** cho cả mẫu, giả định tương quan bất biến.
Thực tế tương quan tăng vọt trong khủng hoảng (bảng theo giai đoạn) nên Pearson **đánh giá thấp rủi ro đúng lúc thị trường căng thẳng**.
Pearson trượt cửa sổ thì phụ thuộc độ dài cửa sổ tuỳ ý và phản ứng trễ; DCC cập nhật mỗi ngày theo cú sốc mới, ước lượng bằng MLE,
đảm bảo $R_t$ luôn xác định dương và được kiểm định là tốt hơn CCC/Static (bảng benchmark).

**Câu 2 – a + b ≈ 1 thì tương quan có đặc điểm gì?** Tương quan có **độ bền cao**: sau cú sốc chung, tương quan tăng và duy trì lâu (half-life
vài tuần), giai đoạn tương quan cao/ thấp xuất hiện thành cụm – giống volatility clustering nhưng cho tương quan. Danh mục không thể trông chờ
đa dạng hoá phục hồi nhanh sau khủng hoảng.